# Демо 2. Шаблони розмови і підготовка даних для SFT

В демо 1 донавчання моделі було повʼязане з заміною голови, а отже частковою зміною архітектури нейронів. В багатьох випадках при донавчанні LLM моделей архітектура не змінюється, а змінюється радше формат введених і виведених данних. Для цього потрібно зрозуміти, **у якому вигляді модель бачить діалог**.
В будь-якому випадку фізично на вхід моделі подається тензор з токенами, а за те як з цих токенів утворюються семантичні повідомлення відповідають **шаблони розмови** (chat templates).
Під час донавчання модель може вивчити новий шаблон розмови.

Донавчання LLM методом SFT виконується на діалогах, відформатованих шаблоном розмови.
У цьому ноутбуці розглянемо, що таке шаблон розмови і як підготувати дані у форматі діалогів;
саме донавчання виконаємо в демо 3.

План:
1. Завантажити base- та instruct-версії SmolLM3.
2. Подивитися, як шаблон розмови перетворює різні типи діалогів на текст.
3. Порівняти відповіді base- та instruct-моделі.
4. Перевірити режими міркування `/think` і `/no_think`.
5. Порівняти шаблони різних моделей.
6. Підготувати датасет для SFT у форматі `messages`.

Демо за мотивами вправ курсу [smol-course](https://huggingface.co/learn/smol-course) і розділу 11
[Hugging Face LLM Course](https://huggingface.co/learn/llm-course/chapter11/1).
Для запуску в Colab оберіть середовище з GPU: *Runtime → Change runtime type → T4 GPU*.

In [ ]:
# Встановлюємо потрібні бібліотеки; -q вимикає зайвий вивід під час встановлення
!pip install -q "transformers>=4.55.0" "datasets" "accelerate"

In [ ]:
import warnings

import torch
from datasets import get_dataset_split_names, load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

if torch.cuda.is_available():
    device = "cuda"
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"Памʼять GPU: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
elif torch.backends.mps.is_available():
    device = "mps"
    print("Apple MPS")
else:
    device = "cpu"
    warnings.warn("Обчислення на CPU: генерація буде дуже повільною, потрібна GPU")

GPU: Tesla T4
Памʼять GPU: 15.6 GB


In [ ]:
# Детальна інформація про GPU NVIDIA: модель, зайнята й вільна памʼять
if device == "cuda":
    !nvidia-smi

Wed Nov 18 15:47:16 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   60C    P8             14W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. Завантаження моделей

[**SmolLM3**](https://huggingface.co/blog/smollm3) — відкрита модель від Hugging Face на 3 млрд параметрів. Вона має дві версії:
- **Base** — лише після попереднього тренування: продовжує текст, але не веде діалог;
- **Instruct** (`SmolLM3-3B`) — донавчена виконувати інструкції й вести діалог.

In [ ]:
base_model_name = "HuggingFaceTB/SmolLM3-3B-Base"
instruct_model_name = "HuggingFaceTB/SmolLM3-3B"

base_tokenizer = AutoTokenizer.from_pretrained(base_model_name)
instruct_tokenizer = AutoTokenizer.from_pretrained(instruct_model_name)

# Ваги SmolLM3 збережено в bfloat16, але T4 не підтримує його апаратно,
# тому завантажуємо у float16.
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name, dtype=torch.float16, device_map="auto"
)
instruct_model = AutoModelForCausalLM.from_pretrained(
    instruct_model_name, dtype=torch.float16, device_map="auto"
)
print("Моделі завантажено")

config.json:   0%|          | 0.00/943 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.4k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/151 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.92k [00:00<?, ?B/s]

... (12 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/326 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/182 [00:00<?, ?B/s]

Моделі завантажено


## 2. Як шаблон розмови форматує діалог

Діалог задається списком повідомлень із ролями:
- `system` — інструкції для моделі (роль, стиль, обмеження);
- `user` — повідомлення користувача;
- `assistant` — відповіді моделі.

Метод `apply_chat_template` перетворює цей список на рядок зі спеціальними токенами.

Параметр `add_generation_prompt`:
- `False` — для передачі в модель завершених діалогів, тобто для **тренувальних даних**;
- `True` — додає в кінець заголовок відповіді асистента, щоб модель почала відповідати; використовується для **інференсу**.

Для SmolLM3 у системне повідомлення можна додати прапорець `/think` або `/no_think`,
який вмикає чи вимикає режим міркування.

In [ ]:
# Різні типи діалогів для порівняння
conversations = {
    "simple_qa": [
        {"role": "system", "content": "/no_think"},
        {"role": "user", "content": "What is machine learning?"},
    ],
    "with_system": [
        {
            "role": "system",
            "content": "You are a helpful AI assistant specialized in explaining technical concepts clearly. /no_think",
        },
        {"role": "user", "content": "What is machine learning?"},
    ],
    "multi_turn": [
        {"role": "system", "content": "You are a math tutor. /no_think"},
        {"role": "user", "content": "What is calculus?"},
        {
            "role": "assistant",
            "content": "Calculus is a branch of mathematics that deals with rates of change and accumulation of quantities.",
        },
        {"role": "user", "content": "Can you give me a simple example?"},
    ],
    "reasoning_task": [
        {"role": "system", "content": "/think"},
        {
            "role": "user",
            "content": "Solve step by step: If a train travels 120 miles in 2 hours, what is its average speed?",
        },
    ],
}

for conv_type, messages in conversations.items():
    print(f"--- {conv_type.upper()} ---")

    # Завершений діалог: так виглядають тренувальні дані
    formatted_complete = instruct_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )
    # Діалог для інференсу: у кінці додано заголовок відповіді асистента
    formatted_prompt = instruct_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    print("Завершений діалог (add_generation_prompt=False):")
    print(formatted_complete)
    print("\nДля інференсу (add_generation_prompt=True):")
    print(formatted_prompt)
    print("\n" + "=" * 50 + "\n")

--- SIMPLE_QA ---
Завершений діалог (add_generation_prompt=False):
<|im_start|>system
## Metadata

Knowledge Cutoff Date: June 2025
Today Date: 18 November 2026
Reasoning Mode: /no_think

## Custom Instructions

You are a helpful AI assistant named SmolLM, trained by Hugging Face.

<|im_start|>user
What is machine learning?<|im_end|>


Для інференсу (add_generation_prompt=True):
<|im_start|>system
## Metadata

Knowledge Cutoff Date: June 2025
Today Date: 18 November 2026
Reasoning Mode: /no_think

## Custom Instructions

You are a helpful AI assistant named SmolLM, trained by Hugging Face.

<|im_start|>user
What is machine learning?<|im_end|>
<|im_start|>assistant
<think>

</think>



--- WITH_SYSTEM ---
Завершений діалог (add_generation_prompt=False):
<|im_start|>system
## Metadata

Knowledge Cutoff Date: June 2025
Today Date: 18 November 2026
Reasoning Mode: /no_think

## Custom Instructions

You are a helpful AI assistant specialized in explaining technical concepts clearly.

<|im_s

**Що варто помітити:**
- Шаблон SmolLM3 сам додає до системного повідомлення блок метаданих (дата відсічення знань, поточна дата, режим міркування)
  і стандартну інструкцію, якщо власної немає. Модель бачить більше, ніж ми написали.
- Кожне повідомлення обгорнуте токенами `<|im_start|>роль` … `<|im_end|>` (формат ChatML).
- З `add_generation_prompt=True` рядок закінчується на `<|im_start|>assistant`: з цього місця модель починає генерувати.

## 3. Base проти Instruct

Ставимо обом моделям одне питання. Base-модель отримує сирий текст без шаблону,
instruct-модель — діалог, відформатований шаблоном розмови.

In [ ]:
def generate(model, tokenizer, text, max_new_tokens=150, temperature=0.7):
    """Генерує продовження тексту й повертає лише нові токени, без промпту."""
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    # Відкидаємо токени промпту: декодуємо лише те, що згенерувала модель
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

In [ ]:
test_prompt = "Explain me, what is quantum computing?"

# Base-модель: сирий текст без шаблону
print("🤖 BASE-МОДЕЛЬ:")
print("-" * 50)
print(generate(base_model, base_tokenizer, test_prompt))

# Instruct-модель: діалог, відформатований шаблоном розмови
instruct_messages = [
    {"role": "system", "content": "/no_think"},
    {"role": "user", "content": test_prompt},
]
instruct_formatted = instruct_tokenizer.apply_chat_template(
    instruct_messages, tokenize=False, add_generation_prompt=True
)
print("\n" + "=" * 50)
print("💬 INSTRUCT-МОДЕЛЬ:")
print("-" * 50)
print(generate(instruct_model, instruct_tokenizer, instruct_formatted))

🤖 BASE-МОДЕЛЬ:
--------------------------------------------------


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


 And how is it useful?

In the present world, we have been working with computers and their functionality for a very long time. In the last few years, we have been working with quantum computers. But many of us don’t know what quantum computing is, and how it works. In this article, I will try to explain you in a very simple way, what is quantum computing? And how it works? Let’s get started.

What is quantum computing?

Quantum computing is a type of computer that uses quantum mechanics to perform calculations. It is a field of study that is still in its early stages, but it has the potential to revolutionize the way we think about computing.

How does quantum computing work?

Quantum computing works by using

💬 INSTRUCT-МОДЕЛЬ:
--------------------------------------------------
Quantum computing is a revolutionary field of computer science that leverages the principles of quantum mechanics to perform computations. Unlike classical computers, which use bits (0s and 1s) to represent an

## 4. Режими міркування `/think` і `/no_think`

SmolLM3 — гібридна модель: у режимі `/think` вона спочатку генерує міркування в блоці `<think>…</think>`,
а потім відповідь; у режимі `/no_think` блок міркувань порожній.

In [ ]:
reasoning_prompts = [
    "What is 15 × 24? Show your work.",
    "A recipe calls for 2 cups of flour for 12 cookies. How much flour is needed for 30 cookies?",
    "If I have $50 and spend $18.75 on lunch and $12.30 on a book, how much money do I have left?",
]

for thinking_flag in ["/no_think", "/think"]:
    print(f"===== Режим: {thinking_flag} =====\n")
    for i, prompt in enumerate(reasoning_prompts, 1):
        messages = [
            {"role": "system", "content": thinking_flag},
            {"role": "user", "content": prompt},
        ]
        formatted = instruct_tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        # Нижча температура робить відповіді на арифметичні задачі стабільнішими
        answer = generate(instruct_model, instruct_tokenizer, formatted, max_new_tokens=300, temperature=0.3)
        print(f"Задача {i}: {prompt}")
        print(f"Відповідь: {answer}")
        print("\n" + "-" * 50 + "\n")

===== Режим: /no_think =====

Задача 1: What is 15 × 24? Show your work.
Відповідь: To solve the multiplication problem 15 × 24, we can break it down into smaller, more manageable steps. 

First, let's consider the multiplication of 15 and 20, which is part of the 24. 

15 × 20 = 300

Next, we need to account for the remaining 4 in the 24. 

15 × 4 = 60

Now, we add the two results together to find the final answer:

300 + 60 = 360

So, 15 × 24 = 360.

--------------------------------------------------

Задача 2: A recipe calls for 2 cups of flour for 12 cookies. How much flour is needed for 30 cookies?
Відповідь: To solve this problem, we need to determine the amount of flour needed per cookie and then scale it up to 30 cookies.

First, let's find out how much flour is needed per cookie. The recipe calls for 2 cups of flour for 12 cookies. So, we divide the total amount of flour by the number of cookies:

2 cups / 12 cookies = 1/6 cup per cookie

Now that we know the amount of flour n

## 5. У кожної моделі свій шаблон

Ролі `system`, `user`, `assistant` — це домовленість на рівні формату `messages`.
У токенах кожна модель записує їх по-своєму. Порівняємо шаблони кількох моделей.

In [ ]:
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Hi!"},
    {"role": "assistant", "content": "Hello! How can I help?"},
]

for name in ["HuggingFaceTB/SmolLM3-3B", "Qwen/Qwen3-0.6B", "microsoft/Phi-3.5-mini-instruct"]:
    tok = AutoTokenizer.from_pretrained(name)
    print(f"===== {name} =====")
    print(tok.apply_chat_template(messages, tokenize=False))
    print()

===== HuggingFaceTB/SmolLM3-3B =====
<|im_start|>system
## Metadata

Knowledge Cutoff Date: June 2025
Today Date: 18 November 2026
Reasoning Mode: /think

## Custom Instructions

You are a helpful assistant.

<|im_start|>user
Hi!<|im_end|>
<|im_start|>assistant
Hello! How can I help?<|im_end|>




config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

... (7 outputs left out)

tokenizer.model: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

===== microsoft/Phi-3.5-mini-instruct =====
<|system|>
You are a helpful assistant.<|end|>
<|user|>
Hi!<|end|>
<|assistant|>
Hello! How can I help?<|end|>
<|endoftext|>



Прапорці режиму міркування також залежать від моделі. У Qwen3, наприклад,
режим вмикається параметром `enable_thinking` у `apply_chat_template`.

In [ ]:
qwen_tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")
question = [{"role": "user", "content": "What is 2 + 2?"}]

for enable_thinking in [True, False]:
    print(f"===== enable_thinking={enable_thinking} =====")
    print(qwen_tokenizer.apply_chat_template(
        question, tokenize=False, add_generation_prompt=True, enable_thinking=enable_thinking
    ))

===== enable_thinking=True =====
<|im_start|>user
What is 2 + 2?<|im_end|>
<|im_start|>assistant

===== enable_thinking=False =====
<|im_start|>user
What is 2 + 2?<|im_end|>
<|im_start|>assistant
<think>

</think>




### А що з base-моделлю?

Base-модель не донавчали на діалогах, тому шаблон у її токенізаторі або відсутній, або модель ніколи не бачила його токенів у ролі розмітки діалогу.
Перед SFT шаблон для base-моделі задають самостійно — найпростіше взяти його з instruct-версії тієї самої моделі.

In [ ]:
print("Шаблон є в base-токенізаторі:  ", base_tokenizer.chat_template is not None)
print("Шаблон є в instruct-токенізаторі:", instruct_tokenizer.chat_template is not None)

Шаблон є в base-токенізаторі:   False
Шаблон є в instruct-токенізаторі: True


## 6. Підготовка датасету для SFT

Для SFT дані зберігають у форматі `messages` — тобто у списку повідомлень із ролями.
Бібліотека TRL (`SFTTrainer`) сама застосовує до нього шаблон розмови під час тренування.

### SmolTalk2

**SmolTalk2** — датасет, на якому донавчали SmolLM3. Він складається з багатьох частин (splits) з різних джерел.
Список частин можна отримати без завантаження самих даних.

In [ ]:
splits = get_dataset_split_names("HuggingFaceTB/smoltalk2", "SFT")
print(f"Кількість частин: {len(splits)}")
for s in splits:
    print(" -", s)

README.md:   0%|          | 0.00/23.5k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/124 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/113 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/113 [00:00<?, ?it/s]

Кількість частин: 25
 - LongAlign_64k_Qwen3_32B_yarn_131k_think
 - OpenThoughts3_1.2M_think
 - aya_dataset_Qwen3_32B_think
 - multi_turn_reasoning_if_think
 - s1k_1.1_think
 - smolagents_toolcalling_traces_think
 - smoltalk_multilingual8_Qwen3_32B_think
 - smoltalk_systemchats_Qwen3_32B_think
 - table_gpt_Qwen3_32B_think
 - LongAlign_64k_context_lang_annotated_lang_6_no_think
 - Mixture_of_Thoughts_science_no_think
 - OpenHermes_2.5_no_think
 - OpenThoughts3_1.2M_no_think_no_think
 - hermes_function_calling_v1_no_think
 - smoltalk_multilingual_8languages_lang_5_no_think
 - smoltalk_smollm3_everyday_conversations_no_think
 - smoltalk_smollm3_explore_instruct_rewriting_no_think
 - smoltalk_smollm3_smol_magpie_ultra_no_think
 - smoltalk_smollm3_smol_rewrite_no_think
 - smoltalk_smollm3_smol_summarize_no_think
 - smoltalk_smollm3_systemchats_30k_no_think
 - table_gpt_no_think
 - tulu_3_sft_personas_instruction_following_no_think
 - xlam_traces_no_think
 - smoltalk_everyday_convs_reasoning_

In [ ]:
# Завантажуємо лише файл однієї частини через data_files: повний датасет займає десятки гігабайт,
# а load_dataset(..., split=...) спершу перебирає файли всіх частин і працює дуже довго
split = "smoltalk_everyday_convs_reasoning_Qwen3_32B_think"
everyday = load_dataset("HuggingFaceTB/smoltalk2", data_files=f"SFT/{split}-*.parquet", split="train")
print(everyday)
print()
for message in everyday[0]["messages"]:
    print(f"[{message['role']}] {message['content'][:200]}")

SFT/smoltalk_everyday_convs_reasoning_Qw(…): reconstructing file:   0%|          |  0.00B / 6.33MB            

SFT/smoltalk_everyday_convs_reasoning_Qw(…): downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['messages', 'chat_template_kwargs', 'source'],
    num_rows: 2057
})

[user] Hi there
[assistant] <think>
Okay, the user sent "Hi there". That's a friendly greeting. I should respond in a welcoming way. Let me check the guidelines. I need to be helpful, keep the conversation going, and maybe ask h
[user] I'm looking for a healthy breakfast idea. What's a good option?
[assistant] <think>
Okay, the user is asking for a healthy breakfast idea. Let me think about what makes a breakfast healthy. It should be balanced, providing a mix of nutrients like protein, fiber, healthy fats,


Ця частина зібрана для режиму `/think`, тому відповіді асистента містять блок міркувань `<think>…</think>`.

Як видно з назви частини, відповіді в ній згенерувала модель Qwen3-32B. Такі **синтетичні дані** від сильнішої моделі —
поширений спосіб отримати тисячі прикладів для SFT без ручного написання відповідей. Помилки моделі-генератора при цьому переходять у дані,
тому синтетичні дані фільтрують.

### Перетворення власного датасету у формат `messages`

Більшість датасетів має інший формат, наприклад колонки «питання» і «відповідь».
Перетворимо датасет задач з математики **GSM8K** у формат `messages`.

In [ ]:
gsm8k = load_dataset("openai/gsm8k", "main", split="train[:100]")  # невелика вибірка для демо
print("Оригінальний приклад:", gsm8k[0])


def to_messages(examples):
    """Перетворює пари «питання — відповідь» на діалоги з ролями."""
    processed = []
    for question, answer in zip(examples["question"], examples["answer"]):
        processed.append([
            {"role": "system", "content": "You are a math tutor. Solve problems step by step."},
            {"role": "user", "content": question},
            {"role": "assistant", "content": answer},
        ])
    return {"messages": processed}


gsm8k_messages = gsm8k.map(to_messages, batched=True, remove_columns=gsm8k.column_names)
print("\nУ форматі messages:", gsm8k_messages[0])

README.md:   0%|          | 0.00/7.93k [00:00<?, ?B/s]

main/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.31MB            

main/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

main/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  419kB            

main/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

Оригінальний приклад: {'question': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?', 'answer': 'Natalia sold 48/2 = <<48/2=24>>24 clips in May.\nNatalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.\n#### 72'}


Map:   0%|          | 0/100 [00:00<?, ? examples/s]


У форматі messages: {'messages': [{'content': 'You are a math tutor. Solve problems step by step.', 'role': 'system'}, {'content': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?', 'role': 'user'}, {'content': 'Natalia sold 48/2 = <<48/2=24>>24 clips in May.\nNatalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.\n#### 72', 'role': 'assistant'}]}


### Що побачить модель під час тренування

Застосуємо шаблон розмови до підготовленого прикладу. Саме цей рядок модель отримає під час SFT.

In [ ]:
text = instruct_tokenizer.apply_chat_template(
    gsm8k_messages[0]["messages"],
    tokenize=False,
    add_generation_prompt=False,  # діалог завершений: відповідь асистента вже є
)
print(text)
print(f"Довжина в токенах: {len(instruct_tokenizer(text)['input_ids'])}")

<|im_start|>system
## Metadata

Knowledge Cutoff Date: June 2025
Today Date: 18 November 2026
Reasoning Mode: /think

## Custom Instructions

You are a math tutor. Solve problems step by step.

<|im_start|>user
Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?<|im_end|>
<|im_start|>assistant
Natalia sold 48/2 = <<48/2=24>>24 clips in May.
Natalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.
#### 72<|im_end|>

Довжина в токенах: 145


**Що варто помітити:** шаблон знову додав метадані й режим `/think`, хоча в даних їх немає,
а відповідь асистента не містить блоку міркувань. Якщо донавчати на таких прикладах, модель вчитиметься
відповідати без міркувань навіть у режимі `/think`. Тому формат даних і шаблон мають узгоджуватися з тим,
як модель використовуватиметься після донавчання.

## Відповіді на теоретичні питання (`02_chat_templates.ipynb`)

### 11. Як довго виконувалися клітинки ноутбуку `02_chat_templates`? Чи були проблеми з запуском, які?
* **Час виконання:** Повне виконання ноутбуку на T4 GPU зайняло близько **4–5 хвилин**. Найбільше часу пішло на завантаження двох моделей по 3 млрд параметрів (`SmolLM3-3B-Base` і `SmolLM3-3B` — кожна важить по 6.15 ГБ, завантаження тривало близько 46 с та 64 с відповідно), а також на генерацію відповідей для 6 математичних запусків у секції 4 (близько 1–1.5 хв).
* **Проблеми та особливості:**
  1. Якби моделі завантажували у стандартному `float32`, обидві (по 12 ГБ кожна) не помістилися б одночасно у 15.6 ГБ відеопам'яті T4. Завдяки `dtype=torch.float16` кожна зайняла лише близько 6 ГБ, тому обидві моделі успішно розмістилися в пам'яті.
  2. Під час генерації з'явилися некритичні попередження `[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer` та попередження про `rope_parameters` для моделі `Phi-3.5-mini-instruct` у секції 5, які не впливають на роботу коду.

### 12. Для чого змінювати параметр `add_generation_prompt` в методі `apply_chat_template()` при використанні моделі? Як по згенерованим розмовам можна дізнатися значення цього параметру, яке було задано при виклику метода?
* **Для чого змінювати:**
  * `add_generation_prompt=False` використовується під час **підготовки тренувальних даних для SFT**, коли діалог уже завершений (містить і запит `user`, і еталонну відповідь `assistant`).
  * `add_generation_prompt=True` використовується під час **інференсу (генерації)**. Він додає в кінець відформатованого рядка стартовий токен репліки асистента, даючи моделі сигнал, що тепер її черга генерувати відповідь.
* **Як розпізнати по відформатованому тексту:**
  * Якщо було задано `add_generation_prompt=False`, рядок закінчується закриваючим токеном останнього повідомлення: `<|im_end|>\n`.
  * Якщо було задано `add_generation_prompt=True`, у самому кінці рядка після `<|im_end|>` додається відкриваючий заголовок відповіді асистента: `<|im_start|>assistant\n` (а в режимі `/no_think` для SmolLM3 ще й порожній блок `<think>\n\n</think>\n\n`).

### 13. Чим відрізняються ролі `system` від `assistant` в шаблонах розмов?
* **`system` (Системна роль):** задає глобальний контекст, правила поведінки, персону (наприклад, `"You are a math tutor"`), обмеження формату або прапорці режиму роботи (`/think`, `/no_think`) перед початком діалогу. Ці інструкції керують моделлю протягом усього діалогу.
* **`assistant` (Роль асистента):** містить безпосередні відповіді самої моделі на запити користувача (`user`). Під час інференсу модель генерує текст саме всередині блоку `assistant`, а під час SFT-тренування ваги моделі оновлюються для мінімізації помилки передбачення саме токенів ролі `assistant`.

### 14. Прокоментуйте вивід даних в секції 2 «Як шаблон розмови форматує діалог». Які секції було додано шаблонізатором розмов і для чого?
* Шаблонізатор Jinja моделі `SmolLM3-3B` автоматично обгортає кожне повідомлення у спеціальні токени формату ChatML (`<|im_start|>роль ... <|im_end|>`) і суттєво розширює системне повідомлення двома структурованими секціями:
  1. **`## Metadata`** — містить поля:
     * `Knowledge Cutoff Date: June 2025` (дата відсічення знань моделі);
     * `Today Date: 18 November 2026` (поточна дата для орієнтації в часі);
     * `Reasoning Mode: /no_think` або `/think` (активний режим міркування).
  2. **`## Custom Instructions`** — містить системний промпт:
     * Якщо користувач не передав власної інструкції (як у `SIMPLE_QA`), шаблон автоматично підставляє дефолтну: *"You are a helpful AI assistant named SmolLM, trained by Hugging Face."*
     * Якщо увімкнено `/think` (як у `REASONING_TASK`), шаблон автоматично додає розгорнуту інструкцію про те, як структурувати відповідь на секції `<think> Thought section </think> Solution section`.
  3. **Порожній блок `<think>\n\n</think>` для асистента в режимі `/no_think`:** коли `add_generation_prompt=True` і режим `/no_think`, шаблон одразу закриває тег `<think></think>`, щоб модель не витрачала токени на роздуми, а негайно писала фінальну відповідь.

### 15. Прокоментуйте різницю між Base і Instruct, проглянувши вивід в секції 3.
* На запит `"Explain me, what is quantum computing?"` моделі реагують принципово по-різному:
  * **`SmolLM3-3B-Base` (базова модель):** сприймає промпт не як запитання, на яке треба дати відповідь, а як початок статті чи вебсторінки, яку треба продовжити. Вона дописує ще одне запитання (*"And how is it useful?"*), потім генерує вступ до блог-посту (*"In this article, I will try to explain you..."*) і починає імітувати структуру статті з підзаголовками.
  * **`SmolLM3-3B` (Instruct-модель):** завдяки SFT та шаблону розмови чітко розпізнає межу запиту користувача (`<|im_start|>user...<|im_end|>`) і з позиції `<|im_start|>assistant` одразу генерує пряму, структуровану відповідь із поясненням квантових обчислень і списком ключових понять (*1. Qubits...*).

### 16. Чому в режимі `/think` при генерації було передано параметр `max_new_tokens=300`?
* У секції 3 використовувалося значення за замовчуванням `max_new_tokens=150`. Проте в режимі `/think` модель спочатку генерує детальні покрокові міркування всередині блоку `<think>...</think>`, і лише після закриваючого тегу `</think>` видає фінальну відповідь. Якби ліміт залишили на рівні 150 токенів, модель гарантовано обірвалася б на півслові ще на початку міркувань. Ліміт збільшили вдвічі (`max_new_tokens=300`), щоб дати моделі більше простору для генерації ланцюжка міркувань (*Chain-of-Thought*).

### 17. Чи впоралася модель з математичними задачами? Який би режим генерації Ви обрали для себе, якщо б Вам потрібна була допомога?
* **Чи впоралася модель:**
  * Усі три задачі (`15 × 24 = 360`, пропорція з борошном `5 cups`, решта грошей `$50 - $18.75 - $12.30 = $18.95`) модель **обчислила абсолютно правильно в обох режимах**.
  * Однак у режимі `/no_think` модель вклалася в ліміт токенів і видала чисті, лаконічні та завершені розв'язки. А в режимі `/think` модель знайшла правильну відповідь всередині блоку `<think>`, але почала перевіряти себе по другому колу (*"Wait, is that right? Let me check my steps again..."*) і **в усіх трьох задачах вичерпала ліміт у 300 токенів**, так і не закривши тег `</think>` і не вивівши фінальний блок `Solution`.
* **Який режим обрати:**
  * Для **простих арифметичних і побутових задач** краще обрати **`/no_think`** — він працює значно швидше, споживає менше токенів і одразу видає чітке покрокове пояснення.
  * Для **складних задач, логічних головоломок або написання коду** варто обирати **`/think`**, але обов'язково збільшувати `max_new_tokens` до `1024–2048`, щоб модель встигла завершити самоперевірку і вивести фінальний результат.

### 18. Які відмінності в шаблонах розмов Ви помітили в секції 5?
* Порівняння `SmolLM3-3B`, `Qwen3-0.6B` та `Phi-3.5-mini-instruct` на одному діалозі показує такі відмінності:
  1. **Спеціальні токени розмітки ролей:**
     * `SmolLM3-3B` та `Qwen3-0.6B` використовують стандарт **ChatML**: `<|im_start|>role\ncontent<|im_end|>`.
     * `Phi-3.5-mini-instruct` використовує власні теги ролей: `<|system|>`, `<|user|>`, `<|assistant|>`, кожне повідомлення закриває токеном `<|end|>`, а в кінці діалогу ставить `<|endoftext|>`.
  2. **Поведінка з системним промптом:**
     * `Qwen3` та `Phi-3.5` передають системне повідомлення `"You are a helpful assistant."` дослівно без жодних доповнень.
     * `SmolLM3` автоматично інжектує в `system` блоки `## Metadata` (дату, cutoff, режим `/think`) та заголовок `## Custom Instructions` (і після блоку `system` у SmolLM3 немає закриваючого `<|im_end|>`).
  3. **Керування режимом міркувань (`thinking`):**
     * У `SmolLM3` режим задається текстовим прапорцем `/think` або `/no_think` у системному промпті.
     * У `Qwen3` режим перемикається булевим аргументом `enable_thinking=True/False` у `apply_chat_template()`. Навіть у завершеному діалозі без міркувань шаблон `Qwen3` автоматично вставляє порожній блок `<think>\n\n</think>\n\n` перед відповіддю асистента.
  4. **Відсутність шаблону в Base-моделі:** у `base_tokenizer.chat_template` значення дорівнює `None` (`False`), тоді як в `instruct_tokenizer` шаблон присутній (`True`).

### 19. Для чого нам потрібно знати про шаблони розмов, якщо ми хочемо донавчити модель за допомогою SFT?
* **Запобігання розбіжності форматів (Train/Inference Mismatch):** Під час SFT модель навчається асоціювати спеціальні токени шаблону з межами реплік і початком генерації. Якщо донавчити модель без шаблону або з неправильним шаблоном, вона не знатиме, коли зупинятися (не генеруватиме токен кінця репліки `<|im_end|>`) і почне продовжувати діалог сама з собою за користувача.
* **Узгодження прихованих інструкцій і режимів (наприклад, `/think`):** Як показано в кінці секції 6 на прикладі GSM8K, шаблон SmolLM3 за замовчуванням автоматично додає в системний промпт `Reasoning Mode: /think`, тоді як відповіді в датасеті GSM8K не містять тегів `<think>...</think>`. Якщо не врахувати це перед SFT, модель навчиться ігнорувати режим `/think` і відповідати без блоку міркувань навіть тоді, коли користувач явно просить її поміркувати.
* **Донавчання Base-моделей:** Оскільки базові моделі взагалі не мають `chat_template`, перед SFT розробник має власноруч призначити їм шаблон (наприклад, скопіювати з Instruct-версії).

### 20. З яких даних було побудовано датасет `HuggingFaceTB/smoltalk2`?
* `SmolTalk2` — це композитний SFT-датасет із **25 спеціалізованих частин (splits)**, який поєднує відкриті датасети та **синтетичні дані**, згенеровані потужнішими моделями (зокрема `Qwen3-32B`). Він охоплює обидва режими роботи (`/think` і `/no_think`) за такими напрямами:
  * **Математика, наука та складні міркування:** `OpenThoughts3_1.2M`, `s1k_1.1_think`, `Mixture_of_Thoughts_science`, `multi_turn_reasoning_if_think`.
  * **Загальні діалоги та виконання інструкцій:** `OpenHermes_2.5`, `smol_magpie_ultra`, `tulu_3_sft_personas_instruction_following`, `everyday_conversations`, `systemchats`.
  * **Робота з текстом (редагування та сумаризація):** `smol_rewrite`, `smol_summarize`, `explore_instruct_rewriting`.
  * **Виклик інструментів та функцій (Tool/Function Calling):** `smolagents_toolcalling_traces_think`, `hermes_function_calling_v1`, `xlam_traces`.
  * **Робота з таблицями та довгим контекстом (до 64k токенів):** `table_gpt`, `LongAlign_64k`.
  * **Багатомовність:** `aya_dataset_Qwen3_32B_think`, `smoltalk_multilingual8`.

### 21. Які модифікації даних відбуваються в секції «Перетворення власного датасету у формат `messages`»?
* У цій секції табличний математичний датасет `openai/gsm8k` трансформується за допомогою функції `to_messages` та методу `.map()`:
  1. **Структурне перетворення колонок у список словників:** Замість двох окремих текстових колонок `question` та `answer` створюється єдина колонка `messages`, де кожен приклад є списком із трьох словників `{"role": ..., "content": ...}`.
  2. **Додавання системного промпту:** До кожного прикладу першим елементом додається фіксоване повідомлення з роллю `system`: `{"role": "system", "content": "You are a math tutor. Solve problems step by step."}`.
  3. **Мапінг ролей:** Текст із поля `question` записується у словник із роллю `"user"`, а покроковий розв'язок із поля `answer` — у словник із роллю `"assistant"`.
  4. **Очищення старих колонок:** Завдяки параметру `remove_columns=gsm8k.column_names` початкові колонки (`question`, `answer`) видаляються, залишаючи чистий датасет зі схемою `{'messages': [...]}`.